# RGB-D feature tracking to quaternion attitude and inertia identification

This notebook is an end-to-end, replaceable-dataset workflow. Each cell performs one stage and displays its result immediately below it.

**Estimator/truth separation:** RGB, depth, masks, camera intrinsics, and timing are processed first. The attitude, angular velocity, and normalized inertia estimate are frozen before the ground-truth section is allowed to load simulation truth.

The operational pose frontend remains the regression-verified **Script 48 SIFT + Lucas-Kanade + RGB-D + RANSAC + weighted-SVD** implementation. The visualization cells can use either SIFT or ORB so their behavior can be compared without silently changing the estimator.

## 1. Inputs — edit only this cell for another simulation

In [ ]:
import os
from pathlib import Path

# ---------------------------------------------------------------------------
# Paths: everything is built from two roots, so nothing machine-specific is committed.
#   REPO      = folder this notebook lives in (Jupyter starts in the notebook's folder)
#   DATA_ROOT = where the rendered datasets live. Set the RGBD_DATA_ROOT environment
#               variable on your machine, otherwise <repo>/data is used.
#   STRIDED_ROOT = where strided copies are written (RGBD_STRIDED_ROOT, else <data>/strided)
# ---------------------------------------------------------------------------
REPO = Path.cwd()

#RGBD_DATA_ROOT = "C:\\Users\\Tevin\\OneDrive - Rensselaer Polytechnic Institute\\URP\\scripts\\blender_sat_final"

DATA_ROOT = Path(os.environ.get("RGBD_DATA_ROOT", REPO / "data"))
STRIDED_ROOT = Path(os.environ.get("RGBD_STRIDED_ROOT", DATA_ROOT / "strided"))

# Dataset: observer eclipse (full sunlit -> penumbra -> umbra -> penumbra -> sunlit pass)
DATASET_observer_eclipse_full = DATA_ROOT / "eclipse_full"
assert DATASET_observer_eclipse_full.exists(), (
    f"Dataset not found: {DATASET_observer_eclipse_full}\n"
    "Set RGBD_DATA_ROOT to the folder that contains 'eclipse_full', or place it in <repo>/data.")

# function for making strided dataset ()
import shutil
FRAME_START = 0
FRAME_END   = 999      # None = last frame
FRAME_STRIDE = 5
BASE_FPS = 10


def make_strided_dataset(src, start, end, stride):
    if start == 0 and end is None and stride == 1: return src
    tag = f"s{start}_e{end if end is not None else 'end'}_x{stride}"
    dst = STRIDED_ROOT / f"{src.name}_{tag}"
    if dst.exists(): return dst
    for sub, pat in [("rgb","rgb_*.png"),("depth","depth_*.npy"),("masks","mask_*.png"),("ground_truth","pose_*.json")]:
        files = sorted((src/sub).glob(pat))
        stop = None if end is None else end + 1
        (dst/sub).mkdir(parents=True)
        for j, f in enumerate(files[start:stop:stride]):
            shutil.copy2(f, dst/sub/f"{f.stem.rstrip('0123456789')}{j:05d}{f.suffix}")
    for name in ["K.npy", "inertial_properties.json"]:
        shutil.copy2(src/name, dst/name)
    return dst

DATASET = make_strided_dataset(DATASET_observer_eclipse_full, FRAME_START, FRAME_END, FRAME_STRIDE)
FPS = BASE_FPS / FRAME_STRIDE


T_INIT = 10.0   # s of data available before the MEKF starts (initial inertia batch)


# Sensor layout. Change these if another simulation uses different directory names.
RGB_SUBDIR = "rgb"                 # e.g. "rgb" or "left_rgb"
DEPTH_SUBDIR = "depth"
MASK_SUBDIR = "masks"
RGB_GLOB = "*.png"
DEPTH_GLOB = "*.npy"
MASK_GLOB = "*.png"

# Calibration: use K.npy when present, otherwise camera_intrinsics.json.
K_NPY = "K.npy"
K_JSON = "camera_intrinsics.json"
#FPS = 10                         # None = read dataset_summary.json; otherwise set numeric FPS
DEPTH_KIND = "Z_DEPTH_METRES"      # this notebook backprojects camera-axis Z depth

# Feature visualization. This does not change the frozen Script 48 estimator.
VISUAL_FEATURE_METHOD = "SIFT"     # "SIFT" or "ORB"
VIDEO_MAX_WIDTH = 960
VIDEO_FPS = 10
THREED_STRIDE = 10                 # one 3D registration view every N frame transitions

# Runtime/caching. Existing frozen attitude is reused when available.
USE_FROZEN_ATTITUDE_IF_AVAILABLE = True
FROZEN_ATTITUDE = (DATASET/"estimated_dynamics"/"R_est_frozen.npy") if FRAME_STRIDE == 1 \
                  else (DATASET/"notebook85_end_to_end"/"R_est_frozen.npy")

# Ground truth is intentionally not opened until Section 9.
GT_DIR = DATASET / "ground_truth"
GT_GLOB = "pose_*.json"
GT_ROTATION_KEY = "rotation_body_to_camera"
GT_OMEGA_KEY = "angular_velocity_body_rad_s"
GT_INERTIA_FILE = DATASET / "inertial_properties.json"
GT_INERTIA_KEY = "inertia_about_com_body_kg_m2"

OUTPUT = DATASET / "notebook85_end_to_end"
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Dataset:", DATASET)
print("Outputs:", OUTPUT)



## 2. Imports and reusable project modules

In [ ]:
import sys, json, csv, importlib, warnings; print(sys.executable)

import cv2, numpy as np, matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt
from scipy.spatial.transform import Rotation
from scipy.signal import savgol_filter
from IPython.display import display, Video, Image, Markdown

SCRIPTS = REPO / "scripts"   # project modules (pose_frontend_script48, notebook_inertia_pipeline, 84_...)
if str(SCRIPTS) not in sys.path: sys.path.insert(0, str(SCRIPTS))
import pose_frontend_script48 as pf
from notebook_inertia_pipeline import run_adjacent
inertia84 = importlib.import_module("84_integral_angular_momentum_inertia")
print("Exact frontend source:", pf._SOURCE)
print("Script 48 parameters:", pf.PARAMETERS)



## 3. Discover synchronized inputs and validate dimensions

In [ ]:
rgb_files=sorted((DATASET/RGB_SUBDIR).glob(RGB_GLOB))
depth_files=sorted((DATASET/DEPTH_SUBDIR).glob(DEPTH_GLOB))
mask_files=sorted((DATASET/MASK_SUBDIR).glob(MASK_GLOB))
N=min(len(rgb_files),len(depth_files),len(mask_files))
assert N>=2, f"Need at least 2 synchronized RGB/depth/mask frames; found {N}"
rgb_files,depth_files,mask_files=rgb_files[:N],depth_files[:N],mask_files[:N]

if (DATASET/K_NPY).exists(): K=np.load(DATASET/K_NPY).astype(float)
else: K=np.asarray(json.loads((DATASET/K_JSON).read_text())["K"],float)
assert K.shape==(3,3)

if FPS is None:
    summary=json.loads((DATASET/"dataset_summary.json").read_text())
    FPS=float(summary["fps"])
DT=1.0/FPS
im0=cv2.imread(str(rgb_files[0])); d0=np.load(depth_files[0]); m0=cv2.imread(str(mask_files[0]),0)
assert im0 is not None and d0.shape[:2]==im0.shape[:2] and m0.shape[:2]==im0.shape[:2]
print(f"{N} frames | {im0.shape[1]}x{im0.shape[0]} | {FPS:g} FPS | dt={DT:g} s")
print("K (3x3)=\n",K)
print("Rotation R: 3x3 | translation/3D point/angular velocity: 3x1 | quaternion: 4x1 XYZW")

In [ ]:
# 3b. per-frame target brightness (sensor-only, used to mark eclipse)
brightness = np.empty(N)
for k, (pr, pm) in enumerate(zip(rgb_files, mask_files)):
    g = cv2.imread(str(pr), cv2.IMREAD_GRAYSCALE).astype(float)
    m = cv2.imread(str(pm), 0) > 0
    brightness[k] = np.percentile(g[m], 95) if m.sum() > 50 else np.percentile(g, 95)  # 95 percentile intensity of pixels of the target

bright_rel = brightness / np.percentile(brightness, 90)          # 1 ≈ fully sunlit
np.save(OUTPUT/"frame_brightness_rel.npy", bright_rel)

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(np.arange(N)/FPS, bright_rel); ax.set(xlabel="Time [s]", ylabel="relative brightness")
ax.grid(alpha=.25); plt.show()

## 4. RGB frames and RGB MP4

+This cell shows representative sensor frames and writes a complete RGB-only MP4. The video is displayed immediately below the cell.

In [ ]:
def fit_width(frame,max_width):
    if frame.shape[1]<=max_width:return frame
    s=max_width/frame.shape[1]
    return cv2.resize(frame,(max_width,int(round(frame.shape[0]*s))),interpolation=cv2.INTER_AREA)

samples=np.linspace(0,N-1,4,dtype=int)
fig,axes=plt.subplots(1,4,figsize=(16,4),constrained_layout=True)
for ax,k in zip(axes,samples):
    bgr=cv2.imread(str(rgb_files[k]));ax.imshow(cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB));ax.set_title(f"Frame {k}");ax.axis("off")
plt.show()

rgb_mp4=OUTPUT/"rgb_sequence.mp4"
first=fit_width(cv2.imread(str(rgb_files[0])),VIDEO_MAX_WIDTH); h,w=first.shape[:2]
vw=cv2.VideoWriter(str(rgb_mp4),cv2.VideoWriter_fourcc(*"mp4v"),VIDEO_FPS,(w,h))
for k,p in enumerate(rgb_files):
    f=fit_width(cv2.imread(str(p)),VIDEO_MAX_WIDTH)
    cv2.putText(f,f"RGB frame {k:03d}",(20,38),cv2.FONT_HERSHEY_SIMPLEX,.9,(255,255,255),2,cv2.LINE_AA)
    vw.write(f)
vw.release()
display(Video(str(rgb_mp4),embed=True,width=min(w,900)))
print(rgb_mp4)

## 5. ORB/SIFT feature tracking, depth backprojection and RANSAC diagnostics

For visualization, the selected detector creates feature seeds. Descriptor matching supplies initial pairs and bidirectional Lucas–Kanade optical flow refines them. Both pixels are backprojected using

\[
X=(u-c_x)Z/f_x,\qquad Y=(v-c_y)Z/f_y.
\]

The exact Script 48 3D RANSAC and weighted-SVD functions classify geometrically consistent correspondences. **Green = RANSAC inlier; red = rejected 3D outlier.**

In [ ]:
def diagnostic_pair(k,method=VISUAL_FEATURE_METHOD):
    b0=cv2.imread(str(rgb_files[k])); b1=cv2.imread(str(rgb_files[k+1]))
    z0=np.load(depth_files[k]).astype(float); z1=np.load(depth_files[k+1]).astype(float)
    ma=cv2.imread(str(mask_files[k]),0); mb=cv2.imread(str(mask_files[k+1]),0)
    # Downscale visualization consistently, including K and depth/masks.
    scale=min(1.0,VIDEO_MAX_WIDTH/b0.shape[1]); size=(int(round(b0.shape[1]*scale)),int(round(b0.shape[0]*scale)))
    b0=cv2.resize(b0,size,interpolation=cv2.INTER_AREA);b1=cv2.resize(b1,size,interpolation=cv2.INTER_AREA)
    z0=cv2.resize(z0,size,interpolation=cv2.INTER_NEAREST);z1=cv2.resize(z1,size,interpolation=cv2.INTER_NEAREST)
    ma=cv2.resize(ma,size,interpolation=cv2.INTER_NEAREST);mb=cv2.resize(mb,size,interpolation=cv2.INTER_NEAREST)
    Ks=K.copy();Ks[0,:]*=scale;Ks[1,:]*=scale
    g0=pf._S48.preprocess_gray(b0);g1=pf._S48.preprocess_gray(b1)
    er0=pf._S48.erode_mask(ma); er1=pf._S48.erode_mask(mb)
    if method.upper()=="ORB": det=cv2.ORB_create(nfeatures=1800);norm=cv2.NORM_HAMMING
    else: det=cv2.SIFT_create(nfeatures=1600,contrastThreshold=.01,edgeThreshold=10);norm=cv2.NORM_L2
    kp0,de0=det.detectAndCompute(g0,er0);kp1,de1=det.detectAndCompute(g1,er1)
    if de0 is None or de1 is None:return None
    raw=cv2.BFMatcher(norm).knnMatch(de0,de1,k=2); good=[a for a,b in raw if a.distance<.75*b.distance]
    if len(good)<8:return None
    p0=np.float32([kp0[m.queryIdx].pt for m in good]); guess=np.float32([kp1[m.trainIdx].pt for m in good])
    lk=dict(winSize=(21,21),maxLevel=3,criteria=(cv2.TERM_CRITERIA_EPS|cv2.TERM_CRITERIA_COUNT,30,.01),flags=cv2.OPTFLOW_USE_INITIAL_FLOW)
    p1,st,_=cv2.calcOpticalFlowPyrLK(g0,g1,p0.reshape(-1,1,2),guess.reshape(-1,1,2),**lk)
    pb,stb,_=cv2.calcOpticalFlowPyrLK(g1,g0,p1,p0.reshape(-1,1,2).copy(),**lk)
    p1=p1.reshape(-1,2);pb=pb.reshape(-1,2); keep=(st.ravel()>0)&(stb.ravel()>0)&(np.linalg.norm(pb-p0,axis=1)<1.5)
    p0,p1=p0[keep],p1[keep]
    def depth_at(z,p):
        q=np.rint(p).astype(int);valid=(q[:,0]>=0)&(q[:,0]<z.shape[1])&(q[:,1]>=0)&(q[:,1]<z.shape[0]);out=np.full(len(p),np.nan);out[valid]=z[q[valid,1],q[valid,0]];return out
    a=depth_at(z0,p0);b=depth_at(z1,p1);valid=np.isfinite(a)&np.isfinite(b)&(a>.01)&(b>.01)&(a<100)&(b<100)
    p0,p1,a,b=p0[valid],p1[valid],a[valid],b[valid]
    if len(p0)<8:return None
    P0=pf.backproject(Ks,p0,a);P1=pf.backproject(Ks,p1,b)
    R,t,inliers,res=pf.ransac_rigid_3d(P0,P1,thresh=pf.PARAMETERS["RANSAC_THRESH_M"],iters=pf.PARAMETERS["RANSAC_ITERS"])
    if R is None:inliers=np.zeros(len(P0),bool);R=np.eye(3);t=np.zeros(3)
    return dict(frame0=b0,frame1=b1,p0=p0,p1=p1,P0=P0,P1=P1,R=R,t=t,inliers=inliers,residual=res)

diag=[]

frame_start = 0

frame_end = N-1 # default is N-1
frame_step = 1
frame_total = int((frame_end - frame_start + 1)/frame_step)

for k in range(frame_start, frame_end, frame_step):
    #print(f"{k}: {VISUAL_FEATURE_METHOD}")
    x=diagnostic_pair(k);diag.append(x)
    
    if ((k+1) - (frame_start+1))%25==0:print(f"diagnostics {int(((k+1)-(frame_start+1))/frame_step)}/{frame_total}")
print("Completed diagnostics using",VISUAL_FEATURE_METHOD)


## 6. Feature tracking and RANSAC-outlier MP4

In [ ]:
track_mp4=OUTPUT/f"feature_tracks_{VISUAL_FEATURE_METHOD.lower()}_ransac.mp4"
sample=next(x for x in diag if x is not None); h,w=sample["frame1"].shape[:2]
vw=cv2.VideoWriter(str(track_mp4),cv2.VideoWriter_fourcc(*"mp4v"),VIDEO_FPS,(w,h))
for k,x in enumerate(diag):
    frame=fit_width(cv2.imread(str(rgb_files[k+1])),VIDEO_MAX_WIDTH)
    if x is not None:
        for a,b,ok in zip(x["p0"],x["p1"],x["inliers"]):
            color=(40,220,80) if ok else (40,40,240)
            aa=tuple(np.rint(a).astype(int));bb=tuple(np.rint(b).astype(int))
            cv2.line(frame,aa,bb,color,1,cv2.LINE_AA);cv2.circle(frame,bb,3,color,-1,cv2.LINE_AA)
        label=f"{VISUAL_FEATURE_METHOD}+LK | green inliers {int(x['inliers'].sum())}/{len(x['inliers'])} | red rejected"
    else:label="insufficient correspondences"
    cv2.rectangle(frame,(0,0),(w,55),(0,0,0),-1);cv2.putText(frame,f"{k:03d}->{k+1:03d}  {label}",(14,35),cv2.FONT_HERSHEY_SIMPLEX,.66,(255,255,255),2,cv2.LINE_AA)
    vw.write(frame)
vw.release();display(Video(str(track_mp4),embed=True,width=min(w,900)));print(track_mp4)

## 7. 3D feature correspondences and registered point-cloud MP4

+Blue points are frame-\(k\) points after applying the estimated rigid transform; orange points are their frame-\(k+1\) partners. Green lines connect inliers and red lines show rejected correspondences.

In [ ]:
cloud_mp4=OUTPUT/"correspondences_3d_ransac.mp4"
frames=[]
for k in range(0, frame_total, THREED_STRIDE):
    x=diag[k]
    if x is None:continue
    transformed=(x["R"]@x["P0"].T).T+x["t"]
    fig=plt.figure(figsize=(9,7));ax=fig.add_subplot(111,projection="3d")
    ax.scatter(transformed[:,0],transformed[:,1],transformed[:,2],s=14,c="tab:blue",label="transformed frame k")
    ax.scatter(x["P1"][:,0],x["P1"][:,1],x["P1"][:,2],s=14,c="tab:orange",label="frame k+1")
    for a,b,ok in zip(transformed,x["P1"],x["inliers"]):ax.plot([a[0],b[0]],[a[1],b[1]],[a[2],b[2]],c="green" if ok else "red",alpha=.45,lw=.8)
    ax.set(title=f"3D correspondences {k}->{k+1}",xlabel="X [m]",ylabel="Y [m]",zlabel="Z [m]");ax.legend(loc="upper right")
    fig.canvas.draw();rgba=np.asarray(fig.canvas.buffer_rgba());frames.append(cv2.cvtColor(rgba,cv2.COLOR_RGBA2BGR));plt.close(fig)
assert frames,"No valid 3D diagnostic frames"
h3,w3=frames[0].shape[:2];vw=cv2.VideoWriter(str(cloud_mp4),cv2.VideoWriter_fourcc(*"mp4v"),2,(w3,h3))
for f in frames:
    for _ in range(2):vw.write(f)
vw.release();display(Video(str(cloud_mp4),embed=True,width=850));print(cloud_mp4)

## 8. Freeze the Script 48 attitude trajectory — no ground truth

In [ ]:
if USE_FROZEN_ATTITUDE_IF_AVAILABLE and FROZEN_ATTITUDE.exists():
    R_est=np.load(FROZEN_ATTITUDE)
    frontend_source=f"cached frozen trajectory: {FROZEN_ATTITUDE}"
    print("if")
else:
    print("else")
    # Exact estimator path. Compound datasets use notebook_inertia_pipeline.run_adjacent;
    # classic Script-48 datasets can instead use pf.run_forward(DATASET).
    run=run_adjacent(DATASET,mode="script48")
    R_est=run["R"]
    frontend_source="exact Script 48 SIFT/LK/RGB-D/RANSAC/SVD rerun"
    np.save(OUTPUT/"R_est_frozen.npy",R_est)
    
    # log how often tracking fails per stride (failure is when 0 inliers found)
    rec = run["records"]

    # save array of valid measurements 
    MIN_INLIERS = 50        # tune from the records
    pair_ok = np.array([r["success"] and r["inliers"] >= MIN_INLIERS for r in rec])
    valid = np.concatenate([[True], pair_ok])  # frame 0 is the reference
    
    valid_strict = np.logical_and.accumulate(valid)
    np.save(OUTPUT/"meas_valid.npy", valid)
    np.save(OUTPUT/"meas_valid_strict.npy", valid_strict)

    
    print(f"stride {FRAME_STRIDE}: {sum(r['success'] for r in rec)}/{len(rec)} pairs ok, median inliers {np.median([r['inliers'] for r in rec])}")
assert len(R_est)==N
q_est=Rotation.from_matrix(R_est).as_quat() # SciPy XYZW
q_est_wxyz = Rotation.from_matrix(R_est).as_quat(scalar_first=True)

# quaternion sign continuity
for k in range(1,len(q_est)):
    if np.dot(q_est[k],q_est[k-1])<0:q_est[k]*=-1

for k in range(1,len(q_est_wxyz)):
    if np.dot(q_est_wxyz[k],q_est_wxyz[k-1])<0:q_est_wxyz[k]*=-1

np.save(OUTPUT/"q_est_frozen_xyzw.npy",q_est)
np.save(OUTPUT/"q_est_frozen_wxyz.npy",q_est_wxyz)

(OUTPUT/"ESTIMATOR_FROZEN_BEFORE_GROUND_TRUTH").write_text(frontend_source)
fig,ax=plt.subplots(figsize=(12,5));ax.plot(np.arange(N)/FPS,q_est);ax.set(xlabel="Time [s]",ylabel="Quaternion component",title="Frozen estimated quaternion trajectory (XYZW)");ax.legend(["qx","qy","qz","qw"]);ax.grid(alpha=.25);plt.show()
print(frontend_source)

# body-frame (B') relative rotation between consecutive frames: R_{k-1}^T R_k
dR = np.einsum('nji,njk->nik', R_est[:-1], R_est[1:])
omega_pair = Rotation.from_matrix(dR).as_rotvec() / DT     # (N-1, 3) rad/s, average rate over [k-1, k]
np.save(OUTPUT/"omega_pair_rad_s.npy", omega_pair)
np.save(OUTPUT/"pair_valid.npy", pair_ok)                   # per pair, NOT cumulative any more

C = np.einsum('nij,nkj->nik', R_est[1:], R_est[:-1])      # R_k R_{k-1}^T, per-pair, camera frame
omega_cam_pair = Rotation.from_matrix(C).as_rotvec() / DT  # ω expressed in the camera (inertial) frame
np.save(OUTPUT/"omega_cam_pair_rad_s.npy", omega_cam_pair)

## 9. Estimate angular velocity from the frozen attitude — no ground truth

In [ ]:
WINDOW = max(5, int(round(21/FRAME_STRIDE)) | 1)   # keep odd

R_mid,omega_est,omega_raw=inertia84.relative_body_rates(R_est,DT,window=WINDOW)
tomega=(np.arange(N-1)+.5)*DT
fig,ax=plt.subplots(figsize=(12,5));ax.plot(tomega,np.degrees(omega_raw),alpha=.2);ax.plot(tomega,np.degrees(omega_est),lw=1.8);ax.set(xlabel="Time [s]",ylabel="Body angular velocity [deg/s]",title="Estimated angular velocity: raw increments and smoothed signal");ax.legend(["raw x","raw y","raw z","smooth x","smooth y","smooth z"],ncol=2);ax.grid(alpha=.25);plt.show()
np.save(OUTPUT/"omega_est_body_rad_s.npy",omega_est)

## 10. Identify normalized inertia — no ground truth

Torque-free inertial angular momentum is constant:

\[
\mathbf H_I=R_{IB,k}\,I_B\,\boldsymbol\omega_{B,k}.
\]

The estimator minimizes the robust residual \(R_{IB,k}I_B\omega_{B,k}-H_I\) over every transition. The principal-moment parameterization guarantees symmetry, positive definiteness, and rigid-body triangle inequalities. Because multiplying \(I\) and \(H\) by the same scalar changes no motion, the result is normalized to \(\operatorname{trace}(I)=1\).

In [ ]:

N_INIT = int(round(T_INIT * FPS)) + 1                 # frames in the init window
assert N_INIT > WINDOW + 2, "init window too short for the rate smoothing window"
R_init = R_est[:N_INIT]

pv = np.load(OUTPUT/"pair_valid.npy")[:N_INIT-1]
print(f"init window: {N_INIT} frames ({T_INIT:g} s), {pv.sum()}/{pv.size} pairs valid")

inertia_fit = inertia84.fit_inertia(R_init, DT, window=WINDOW, starts=32)
I_est = inertia_fit["I"]
draws = inertia84.bootstrap(R_init, DT, inertia_fit, window=WINDOW, samples=100,
                            block=max(3, (N_INIT-1)//8))   # block must be << number of transitions

np.save(OUTPUT/"normalized_inertia_estimate_init.npy", I_est)
np.save(OUTPUT/"inertia_bootstrap_init.npy", draws)
np.save(OUTPUT/"N_INIT.npy", N_INIT)


J_est, V_est = np.linalg.eigh(I_est)      # ascending principal moments + principal axes (columns, in B')
if np.linalg.det(V_est) < 0:
    V_est[:, 2] *= -1                     # proper rotation, so the MEKF can turn it into a quaternion
np.save(OUTPUT/"principal_moments_init.npy", J_est)
np.save(OUTPUT/"principal_axes_init.npy", V_est)


# (keep the rest of the cell; update the json name/fields to include "T_init_s": T_INIT)



"""inertia_fit=inertia84.fit_inertia(R_est,DT,window=WINDOW,starts=32)
I_est=inertia_fit["I"]
draws=inertia84.bootstrap(R_est,DT,inertia_fit,window=WINDOW,samples=100,block=12)
np.save(OUTPUT/"normalized_inertia_estimate.npy",I_est)np.save(OUTPUT/"inertia_bootstrap.npy",draws)"""
frozen={"ground_truth_used":False,"I_trace_one":I_est.tolist(),"principal_moments":np.linalg.eigvalsh(I_est).tolist(),"relative_H_rms":inertia_fit["relative_H_rms"],"jacobian_condition":inertia_fit["jacobian_condition"]}
(OUTPUT/"inertia_estimate_frozen_no_ground_truth.json").write_text(json.dumps(frozen,indent=2))
print("Normalized inertia estimate:\n",I_est)
print("Principal moments:",np.linalg.eigvalsh(I_est))
print("Relative angular-momentum RMS:",inertia_fit["relative_H_rms"])
print("Jacobian condition number:",inertia_fit["jacobian_condition"])

## 11. Evaluation-only ground-truth loader

+This is the first cell permitted to open pose or inertia truth. All estimator products above have already been frozen.

In [ ]:
assert (OUTPUT/"ESTIMATOR_FROZEN_BEFORE_GROUND_TRUTH").exists()
assert (OUTPUT/"inertia_estimate_frozen_no_ground_truth.json").exists()
gt_files=sorted(GT_DIR.glob(GT_GLOB))[:N]
assert len(gt_files)==N,f"Expected {N} truth frames, found {len(gt_files)}"
gt_rows=[json.loads(p.read_text()) for p in gt_files]
R_gt_abs=np.asarray([r[GT_ROTATION_KEY] for r in gt_rows],float)
R_gt=np.asarray([R@R_gt_abs[0].T for R in R_gt_abs])
q_gt=Rotation.from_matrix(R_gt).as_quat()
for k in range(1,len(q_gt)):
    if np.dot(q_gt[k],q_gt[k-1])<0:q_gt[k]*=-1
    
q_gt_wxyz=np.array([q_gt[:,3], q_gt[:,0], q_gt[:,1], q_gt[:,2]])

if GT_OMEGA_KEY and GT_OMEGA_KEY in gt_rows[0]:omega_gt=np.asarray([r[GT_OMEGA_KEY] for r in gt_rows[:-1]],float)
else:_,omega_gt,_=inertia84.relative_body_rates(R_gt,DT,window=1)
props=json.loads(GT_INERTIA_FILE.read_text());I_true=np.asarray(props[GT_INERTIA_KEY],float);I_true/=np.trace(I_true)
print("Ground truth loaded for evaluation only.")

print(I_true)

## 11.5 Output Quaternion GT and Ang. Velocity GT to MEKF

full-rate (10 Hz) truth for the MEKF, built from the UNSTRIDED source dataset (evaluation only)


In [ ]:
src_gt = sorted((DATASET_observer_eclipse_full/"ground_truth").glob(GT_GLOB))[FRAME_START:FRAME_END+1]
rows = [json.loads(p.read_text()) for p in src_gt]
R_abs = np.asarray([r[GT_ROTATION_KEY] for r in rows], float)          # body -> camera
R_rel = np.asarray([R @ R_abs[0].T for R in R_abs])                      # B' -> C, same convention as q_gt
q_full = Rotation.from_matrix(R_rel).as_quat(scalar_first=True)          # wxyz
for k in range(1, len(q_full)):
    if q_full[k] @ q_full[k-1] < 0: q_full[k] *= -1                       # sign continuity
omega_B = np.asarray([r[GT_OMEGA_KEY] for r in rows], float)
omega_cam_full = np.einsum("nij,nj->ni", R_abs, omega_B)                  # body rate expressed in camera frame
np.save(OUTPUT/"q_gt_cam0_wxyz.npy", q_full)
np.save(OUTPUT/"omega_cam0_rad_s.npy", omega_cam_full)

## 12. Quaternion estimate compared with ground truth

In [ ]:
att_err=np.degrees(Rotation.from_matrix(np.asarray([a@b.T for a,b in zip(R_est,R_gt)])).magnitude())

err_rotvec = np.degrees(Rotation.from_matrix([b.T @ a for a, b in zip(R_est, R_gt)]).as_rotvec())       # (N, 3), x/y/z error components, deg
per_axis_rmse = np.sqrt(np.mean(err_rotvec**2, axis=0))   # 3 numbers, diag(R) candidates
att_variance = (np.deg2rad(per_axis_rmse))**2

fig,axes=plt.subplots(5,1,figsize=(13,12),sharex=True,constrained_layout=True)
for j,label in enumerate(["qx","qy","qz","qw"]):axes[j].plot(np.arange(N)/FPS,q_gt[:,j],"k",label="truth");axes[j].plot(np.arange(N)/FPS,q_est[:,j],label="estimate");axes[j].set_ylabel(label);axes[j].grid(alpha=.25)
axes[0].legend();axes[4].plot(np.arange(N)/FPS,att_err,color="tab:red");axes[4].set(xlabel="Time [s]",ylabel="deg",title="SO(3) attitude error");axes[4].grid(alpha=.25);plt.show()
#print(f"Attitude RMSE {np.sqrt(np.mean(att_err**2)):.3f} deg | final {att_err[-1]:.3f} deg")

#print(f"Attitude variance: {att_variance} rad^2 (use as measurement noise in MEKF)")

## 13. Angular velocity estimate compared with ground truth

In [ ]:
m=min(len(omega_est),len(omega_gt));labels=["x","y","z"]
fig,axes=plt.subplots(3,1,figsize=(13,9),sharex=True,constrained_layout=True)
for j in range(3):
    axes[j].plot(tomega[:m],np.degrees(omega_gt[:m,j]),"k",lw=2,label="ground truth")
    axes[j].plot(tomega[:m],np.degrees(omega_est[:m,j]),color=f"C{j}",label="estimate")
    axes[j].set_ylabel(f"omega {labels[j]} [deg/s]");axes[j].grid(alpha=.25);axes[j].legend()
axes[-1].set_xlabel("Time [s]");fig.suptitle("Body angular velocity: estimator versus ground truth");plt.show()
mae=np.degrees(np.mean(np.abs(omega_est[:m]-omega_gt[:m]),axis=0));print("Component MAE [deg/s]:",mae)

## 14. Inertia matrix and principal-moment ratios: prediction versus actual

In [ ]:
err=np.linalg.norm(I_est-I_true)/np.linalg.norm(I_true)
lim=max(np.abs(I_est).max(),np.abs(I_true).max())
fig,axes=plt.subplots(1,2,figsize=(11,4.7),constrained_layout=True)
for ax,(name,M) in zip(axes,[("Estimated",I_est),("Actual (evaluation only)",I_true)]):
    im=ax.imshow(M,cmap="coolwarm",vmin=-lim,vmax=lim)
    for i in range(3):
        for j in range(3):ax.text(j,i,f"{M[i,j]:.3f}",ha="center",va="center")
    ax.set(title=name,xticks=range(3),yticks=range(3))
fig.colorbar(im,ax=axes.tolist(),label="trace-normalized inertia");plt.show()

Je=np.linalg.eigvalsh(I_est);Jt=np.linalg.eigvalsh(I_true)
fig,ax=plt.subplots(figsize=(9,5));x=np.arange(3);w=.34
ax.bar(x-w/2,Je/Je.min(),w,label="estimated");ax.bar(x+w/2,Jt/Jt.min(),w,label="actual")
ax.set(xticks=x,xticklabels=["J1","J2","J3"],ylabel="principal moment / smallest moment",title="Principal inertia ratios: prediction versus actual");ax.grid(axis="y",alpha=.25);ax.legend();plt.show()
print(f"Normalized inertia relative Frobenius error: {100*err:.2f}%")
print("Estimated ratios:",Je/Je.min());print("Actual ratios:",Jt/Jt.min())

## 15. Bootstrap uncertainty and final summary

In [ ]:
mom=np.linalg.eigvalsh(draws)
fig,ax=plt.subplots(figsize=(9,5));ax.boxplot([mom[:,i] for i in range(3)],tick_labels=["J1","J2","J3"]);ax.scatter(range(1,4),Je,c="tab:red",label="full estimate");ax.scatter(range(1,4),Jt,c="black",marker="x",s=70,label="actual");ax.set(title="Moving-block bootstrap uncertainty",ylabel="principal moment / trace(I)");ax.grid(axis="y",alpha=.25);ax.legend();plt.show()
summary={"ground_truth_used_by_estimator":False,"frames":N,"fps":FPS,"attitude_rmse_deg":float(np.sqrt(np.mean(att_err**2))),"final_attitude_error_deg":float(att_err[-1]),"omega_component_mae_deg_s":mae.tolist(),"normalized_inertia_relative_error":float(err),"estimated_I_trace_one":I_est.tolist(),"actual_I_trace_one":I_true.tolist(),"relative_H_rms":inertia_fit["relative_H_rms"],"jacobian_condition":inertia_fit["jacobian_condition"],"rgb_video":str(rgb_mp4),"tracking_video":str(track_mp4),"correspondence_3d_video":str(cloud_mp4)}
(OUTPUT/"evaluation_summary.json").write_text(json.dumps(summary,indent=2));display(summary)

## Interpretation

- The visual frontend observes pose, not inertia directly.
- Angular velocity comes from the logarithm of consecutive rotation matrices.
- The inertia fit uses constant inertial angular momentum instead of differentiating noisy angular velocity.
- Only normalized inertia is observable from torque-free attitude alone.
- A large bootstrap spread or Jacobian condition number means that the motion did not sufficiently excite all inertia directions.
- The ground-truth comparison evaluates the frozen outputs; it never corrects the estimator.

## 16. Data needed for MEKF

In [ ]:
np.save(OUTPUT/"I_true.npy", I_true)
#np.save(OUTPUT/"I_est.npy", I_est)

np.save(OUTPUT/"q_est_wxyz.npy",q_est_wxyz)
np.save(OUTPUT/"q_gt_wxyz.npy",q_gt_wxyz)

